# Geology Forecast Challenge — pipeline avancé

Ce notebook est la suite de `baseline_trend_forecast.ipynb` (persistance / extrapolation linéaire, sans dépendance). Ici on empile des techniques nettement plus avancées, pensées pour faire progresser le classement :

1. **Augmentation par fenêtres glissantes groupées par puits** — beaucoup plus de données d'entraînement que les 1510 lignes de `train.csv`, tout en gardant la traçabilité du puits d'origine.
2. **Validation croisée groupée (`GroupKFold` par puits)** — élimine la fuite de données du split aléatoire utilisé dans le baseline.
3. **Features enrichies** — masque de valeurs manquantes, longueur d'historique réel, pente locale, encodage positionnel sinusoïdal.
4. **Modèle Conv1D + BiLSTM + auto-attention**, entraîné avec une **fonction de coût pondérée** qui imite la métrique du classement (poids plus forts sur les pas proches).
5. **Ensembling** par bagging des modèles des 5 plis de validation croisée.
6. **Incertitude par MC-Dropout** pour générer 9 réalisations réellement différentes (pas des copies, contrairement à `public-11st-private-4th.ipynb`).

⚠️ **Limite honnête** : cet environnement local (Python 3.14 32 bits) ne peut pas installer pandas / numpy / TensorFlow (aucun binaire précompilé disponible, et pas de compilateur pour construire depuis les sources). J'ai donc :
- validé toute la logique d'augmentation et de fenêtrage en Python pur, sur les vraies données de `train_raw/` (123 puits → **48 421 fenêtres** avec les réglages ci-dessous, contre 1510 lignes dans `train.csv`) ;
- vérifié la syntaxe de chaque cellule de ce notebook ;
- mais **je n'ai pas pu exécuter les cellules TensorFlow ici**. Ce notebook est prévu pour tourner sur Kaggle (comme `public-11st-private-4th.ipynb`), avec accélérateur GPU activé.

## 0. Imports et configuration

In [ ]:
import glob
import hashlib
import os

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = "."
RAW_DIR = os.path.join(DATA_DIR, "train_raw")

HIST_LEN = 300        # positions -299 .. 0 -> 300 points connus
FUT_LEN = 300          # positions 1 .. 300 -> 300 points a predire
N_ALT_REALIZATIONS = 9 # r_1_pos_* ... r_9_pos_*

STRIDE = 15            # pas de la fenetre glissante -> densite de l'augmentation
MIN_HIST = 50          # nombre minimal de points d'historique reels par fenetre

N_FOLDS = 5
EPOCHS = 40
BATCH_SIZE = 64

## 1. Augmentation par fenêtres glissantes, groupées par puits

Le script fourni avec le projet (`interpolate_and_split.py`) découpe chaque puits en quelques morceaux **non chevauchants**, ce qui donne 1510 lignes au total pour 123 puits. Ici on fait glisser une fenêtre le long de chaque puits rééchantillonné avec un pas de `STRIDE` points : beaucoup plus d'exemples d'entraînement à partir des mêmes puits (augmentation de données), tout en gardant `well_id` pour chaque ligne — ce qui permettra un vrai split de validation *par puits* juste après.

Chaque fenêtre est recalée comme dans `train.csv` : on soustrait la valeur du point "présent" pour que l'historique se termine toujours à `0.0`.

In [ ]:
def read_well(path):
    df = pd.read_csv(path).sort_values("VS_APPROX_adjusted")
    return df["VS_APPROX_adjusted"].to_numpy(), df["HORIZON_Z_adjusted"].to_numpy()


def resample_uniform(xs, ys, step=1.0):
    grid = np.arange(xs[0], xs[-1] + step, step)
    return np.interp(grid, xs, ys)


def make_windows(values, well_id, stride=STRIDE, min_hist=MIN_HIST):
    n = len(values)
    p_max = n - FUT_LEN - 1
    rows = []
    for p in range(min_hist - 1, p_max + 1, stride):
        hist_start = max(0, p - (HIST_LEN - 1))
        history_raw = values[hist_start:p + 1]
        pad_len = HIST_LEN - len(history_raw)
        anchor = values[p]

        history = np.full(HIST_LEN, np.nan, dtype=np.float32)
        history[pad_len:] = (history_raw - anchor).astype(np.float32)
        future = (values[p + 1:p + 1 + FUT_LEN] - anchor).astype(np.float32)

        gid = "g_" + hashlib.md5(f"{well_id}_{p}".encode("utf-8")).hexdigest()[:10]
        rows.append((well_id, gid, HIST_LEN - pad_len, history, future))
    return rows


all_rows = []
for path in sorted(glob.glob(os.path.join(RAW_DIR, "*.csv"))):
    well_id = os.path.splitext(os.path.basename(path))[0]
    xs, ys = read_well(path)
    resampled = resample_uniform(xs, ys)
    if len(resampled) < MIN_HIST + FUT_LEN:
        continue
    all_rows.extend(make_windows(resampled, well_id))

well_ids = np.array([r[0] for r in all_rows])
geology_ids = np.array([r[1] for r in all_rows])
n_real_hist = np.array([r[2] for r in all_rows], dtype=np.float32)
X_hist = np.stack([r[3] for r in all_rows])   # (N, 300), NaN = valeur manquante
y_future = np.stack([r[4] for r in all_rows]) # (N, 300)

print(f"{len(all_rows)} fenetres generees depuis {len(set(well_ids))} puits")
print("X_hist:", X_hist.shape, "y_future:", y_future.shape)

## 2. Features enrichies

- **valeur + masque** : au lieu de remplacer les `NaN` par une seule valeur (ambigu : est-ce un vrai zéro ou une absence de mesure ?), on donne au réseau la valeur (0 si manquante) **et** un canal binaire "ce point est-il réellement connu ?".
- **longueur d'historique réel** (normalisée) : combien de points sont réellement connus — un signal utile pour que le modèle sache à quel point faire confiance à l'historique.
- **pente locale** : tendance récente (sécante sur les ~30 derniers points connus), le même signal que le baseline utilisait, mais ici comme *feature* plutôt que comme modèle final.
- **encodage positionnel sinusoïdal** : comme dans un Transformer, pour que le modèle sache "à quelle distance dans le passé" se trouve chaque point de la séquence.

In [ ]:
def compute_slope_feature(X, tail=30):
    n_rows = X.shape[0]
    slopes = np.zeros(n_rows, dtype=np.float32)
    for i in range(n_rows):
        idx = np.where(~np.isnan(X[i]))[0]
        if len(idx) < 2:
            continue
        tail_idx = idx[-tail:] if len(idx) > tail else idx
        x0, x1 = tail_idx[0], tail_idx[-1]
        if x1 == x0:
            continue
        slopes[i] = (X[i, x1] - X[i, x0]) / (x1 - x0)
    return slopes


def sinusoidal_positional_encoding(seq_len, n_freq=8):
    positions = np.arange(seq_len, dtype=np.float32)
    pe = np.zeros((seq_len, 2 * n_freq), dtype=np.float32)
    for k in range(n_freq):
        freq = 1.0 / (10000 ** (2 * k / (2 * n_freq)))
        pe[:, 2 * k] = np.sin(positions * freq)
        pe[:, 2 * k + 1] = np.cos(positions * freq)
    return pe


POS_ENCODING = sinusoidal_positional_encoding(HIST_LEN)


def build_features(X_hist_arr, n_real_hist_arr):
    mask = (~np.isnan(X_hist_arr)).astype(np.float32)
    values = np.nan_to_num(X_hist_arr, nan=0.0).astype(np.float32)
    slope = compute_slope_feature(X_hist_arr)

    seq = np.stack([values, mask], axis=-1)                          # (N, 300, 2)
    pe_tiled = np.tile(POS_ENCODING[None, :, :], (len(X_hist_arr), 1, 1))  # (N, 300, 2*n_freq)
    seq_full = np.concatenate([seq, pe_tiled], axis=-1).astype(np.float32)

    aux = np.stack([n_real_hist_arr / HIST_LEN, slope], axis=-1).astype(np.float32)  # (N, 2)
    return seq_full, aux


X_seq, X_aux = build_features(X_hist, n_real_hist)
print("X_seq:", X_seq.shape, "X_aux:", X_aux.shape)

## 3. Fonction de coût pondérée

Le classement Kaggle utilise une variante pondérée du MSE (poids plus forts sur les pas proches, cohérent avec le géosteering où l'erreur immédiatement devant le foret compte plus que l'erreur à 300 pas). La formule exacte n'était pas accessible depuis mon environnement (page *Evaluation* rendue en JavaScript) — `WEIGHTS` ci-dessous est un **proxy à décroissance linéaire**, à remplacer par la vraie pondération si vous la trouvez dans l'onglet *Evaluation* de la compétition.

In [ ]:
def make_weight_vector(n=FUT_LEN, w_min=0.2):
    return np.linspace(1.0, w_min, n).astype(np.float32)


WEIGHTS = make_weight_vector()
WEIGHTS_TF = tf.constant(WEIGHTS)


def weighted_mse(y_true, y_pred):
    diff = tf.square(y_true - y_pred) * WEIGHTS_TF
    return tf.reduce_mean(diff)

## 4. Modèle : Conv1D + BiLSTM + auto-attention

- **Conv1D** (deux couches, noyau causal) : capte les motifs locaux à plusieurs échelles avant même d'entrer dans la partie récurrente.
- **BiLSTM** : mémoire à long terme dans les deux sens de la séquence connue.
- **Multi-Head Attention** (auto-attention sur la sortie du BiLSTM) : laisse le modèle pondérer directement les points d'historique les plus informatifs, plutôt que de tout faire passer par le dernier état caché du LSTM comme dans `public-11st-private-4th.ipynb`.
- Les features auxiliaires (longueur d'historique, pente) sont injectées juste avant les couches denses finales.

Cette architecture s'inspire de l'approche publiée comme solution gagnante de cette compétition (Conv1D + attention + encodage positionnel).

In [ ]:
def build_model(seq_len, n_seq_feat, n_aux, fut_len=FUT_LEN, dropout=0.2):
    seq_in = keras.Input(shape=(seq_len, n_seq_feat), name="seq")
    aux_in = keras.Input(shape=(n_aux,), name="aux")

    x = layers.Conv1D(64, 7, padding="causal", activation="relu")(seq_in)
    x = layers.Conv1D(64, 5, padding="causal", activation="relu")(x)
    x = layers.Bidirectional(layers.LSTM(96, return_sequences=True, dropout=dropout))(x)

    attn = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = layers.LayerNormalization()(x + attn)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Concatenate()([x, aux_in])
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation="relu")(x)
    out = layers.Dense(fut_len, name="point_forecast")(x)

    model = keras.Model(inputs=[seq_in, aux_in], outputs=out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=weighted_mse)
    return model


build_model(HIST_LEN, X_seq.shape[-1], X_aux.shape[-1]).summary()

## 5. Validation croisée groupée par puits + ensembling

On veut garantir que toutes les fenêtres d'un même puits restent du même côté du split (entraînement **ou** validation, jamais les deux) — ça corrige la fuite potentielle du split aléatoire utilisé dans `baseline_trend_forecast.ipynb`. Plutôt que d'ajouter scikit-learn comme dépendance pour ça (une seule fonction utilisée), on réimplémente la logique de `GroupKFold` en quelques lignes : mélanger les puits, les répartir en `N_FOLDS` paquets, puis découper les lignes selon le paquet de leur puits. Testé séparément sur des groupes de tailles variées (aucun chevauchement de puits entre train et validation, plis équilibrés).

Entraîner un modèle par pli donne aussi un ensemble de 5 modèles gratuitement : leurs prédictions moyennées (bagging) au moment de la soumission réduisent la variance par rapport à un modèle unique.

In [ ]:
def group_k_fold(groups, n_splits, seed=SEED):
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_groups)
    fold_of_group = {g: i % n_splits for i, g in enumerate(unique_groups)}
    fold_assignment = np.array([fold_of_group[g] for g in groups])
    for k in range(n_splits):
        va_idx = np.where(fold_assignment == k)[0]
        tr_idx = np.where(fold_assignment != k)[0]
        yield tr_idx, va_idx


fold_models = []
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(group_k_fold(well_ids, N_FOLDS)):
    print(f"--- Pli {fold + 1}/{N_FOLDS} ({len(tr_idx)} train / {len(va_idx)} val) ---")
    model = build_model(HIST_LEN, X_seq.shape[-1], X_aux.shape[-1])
    early_stop = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)

    model.fit(
        {"seq": X_seq[tr_idx], "aux": X_aux[tr_idx]}, y_future[tr_idx],
        validation_data=({"seq": X_seq[va_idx], "aux": X_aux[va_idx]}, y_future[va_idx]),
        epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=2,
    )

    val_pred = model.predict({"seq": X_seq[va_idx], "aux": X_aux[va_idx]}, verbose=0)
    score = float(np.sqrt(np.mean(((y_future[va_idx] - val_pred) ** 2) * WEIGHTS)))
    print(f"pli {fold + 1} — RMSE pondere de validation : {score:.4f}")

    fold_scores.append(score)
    fold_models.append(model)

print(f"\nRMSE pondere moyen (CV) : {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")
print("A comparer directement au RMSE non pondere du baseline (2.83) — les deux metriques ne sont pas identiques, mais donnent un ordre d'idee.")

## 6. Incertitude par MC-Dropout pour les 9 réalisations

Les couches `Dropout` et le `dropout` du BiLSTM restent normalement inactifs à l'inférence. En forçant `training=True` au moment de la prédiction, on garde ce bruit actif : chaque appel au modèle tire un sous-réseau légèrement différent et produit une trajectoire légèrement différente. En combinant ça avec un tirage aléatoire parmi les 5 modèles de l'ensemble, on obtient 9 réalisations qui reflètent deux sources d'incertitude réelles (le désaccord entre modèles, et l'incertitude propre à chaque modèle) — pas 9 copies de la même prévision comme dans `public-11st-private-4th.ipynb`.

In [ ]:
def mc_dropout_samples(models, seq, aux, n_samples=N_ALT_REALIZATIONS, seed=SEED):
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_samples):
        model = models[rng.integers(len(models))]
        pred = model([seq, aux], training=True).numpy()
        samples.append(pred)
    return samples

## 7. Prévision finale et soumission

- **Prévision ponctuelle** (`1...300`) : moyenne des 5 modèles de l'ensemble — c'est la prévision la plus fiable qu'on puisse produire.
- **9 réalisations alternatives** (`r_1_pos_*...r_9_pos_*`) : échantillons MC-Dropout, décrits ci-dessus.

On relit l'en-tête de `sample_submission.csv` pour garantir exactement le même ordre de colonnes que le baseline.

In [ ]:
hist_cols = [str(c) for c in range(-299, 1)]

test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_ids = test_df["geology_id"].to_numpy()
X_test_hist = test_df[hist_cols].to_numpy(dtype=np.float32)
n_real_hist_test = (~np.isnan(X_test_hist)).sum(axis=1).astype(np.float32)
X_test_seq, X_test_aux = build_features(X_test_hist, n_real_hist_test)

fold_preds = [m.predict({"seq": X_test_seq, "aux": X_test_aux}, verbose=0) for m in fold_models]
point_forecast = np.mean(fold_preds, axis=0)

alt_preds = mc_dropout_samples(fold_models, X_test_seq, X_test_aux)

sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

# on construit toutes les colonnes dans un dict d'abord, puis un seul DataFrame
# d'un coup (inserer 3000 colonnes une par une fragmenterait le DataFrame)
columns = {"geology_id": test_ids}
for i in range(FUT_LEN):
    columns[str(i + 1)] = point_forecast[:, i]
for r, alt in enumerate(alt_preds, start=1):
    for i in range(FUT_LEN):
        columns[f"r_{r}_pos_{i + 1}"] = alt[:, i]

submission = pd.DataFrame(columns)[sample_sub.columns]

assert list(submission.columns) == list(sample_sub.columns), "colonnes differentes de sample_submission.csv"
assert list(submission["geology_id"]) == list(sample_sub["geology_id"]), "ordre des geology_id different"
assert not submission.drop(columns=["geology_id"]).isna().any().any(), "des NaN se sont glisses dans la soumission"

submission.to_csv(os.path.join(DATA_DIR, "submission_advanced.csv"), index=False)
print("submission_advanced.csv ecrit avec succes.")

## Bilan et pistes non implémentées ici

**Ce qui a été ajouté par rapport au baseline** : augmentation par fenêtres glissantes (~32× plus de données), split par puits sans fuite, masque de valeurs manquantes + longueur d'historique + pente locale + encodage positionnel comme features, modèle Conv1D + BiLSTM + attention entraîné avec une perte pondérée, ensembling à 5 plis, incertitude par MC-Dropout pour des réalisations réellement diverses.

**Pistes plus avancées, volontairement pas codées ici** (pour ne pas livrer du code que je n'ai pas pu tester) :

- **Apprentissage multi-hypothèses (Winner-Takes-All)** : au lieu de 10 réalisations bricolées après coup, entraîner directement 10 têtes de sortie avec une perte qui ne rétropropage que sur la tête la plus proche de la vérité à chaque exemple (+ un petit terme sur la moyenne des têtes pour éviter que certaines ne soient jamais entraînées). C'est la technique standard pour la prédiction de trajectoires multiples en robotique/conduite autonome, et elle collerait particulièrement bien au format à 10 réalisations de cette compétition.
- **Poids réels de la métrique** : si vous retrouvez le `k_matrix` exact dans l'onglet *Evaluation* de Kaggle, remplacez `make_weight_vector()` — l'écart entre le proxy et la vraie métrique est la plus grande source d'incertitude sur le classement final.
- **Plus d'augmentation** : réduire `STRIDE`, ou faire varier aussi la longueur de la fenêtre (comme `interpolate_and_split.py` le fait avec des chunks de 350 à 600) plutôt qu'une fenêtre de taille fixe.
- **Test-time augmentation** : moyenner les prédictions sur plusieurs longueurs d'historique tronqué au moment de l'inférence.
- **Recherche d'hyperparamètres** (taille des couches, nombre de têtes d'attention, `STRIDE`, `EPOCHS`) une fois qu'un premier score de leaderboard est disponible pour orienter les choix.